Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import os
import shutil

# Step 1: Ensure 'data' directory exists
if not os.path.exists("data"):
    os.makedirs("data")

# Step 2: Run the SQL query and get the results as a DataFrame
query = """
select dt, avg(csat)
from(
with csat_new as (
select  sub_ticket_id,
max_by(case when response_value in ('1','2','3','4','5','1.0','2.0','3.0','4.0','5.0') then try_cast(response_value as decimal) end,case when response_value in ('1','2','3','4','5','1.0','2.0','3.0','4.0','5.0') then modified_at end) as csat,
array_agg(case when response_value not in ('1','2','3','4','5','1.0','2.0','3.0','4.0','5.0') then response_value end) as csat_free_text, 
max(case when cast(question_id AS int) = 9 and response_value in ('Yes','అవును','ஆம்','होय','हाँ','ಹೌದು','അതെ' )  then 'RESOLVED'
when cast(question_id AS int) = 9 and response_value in ('No','సంఖ్య','இல்லை','नाही','नहीं','ಇಲ್ಲ','ഇല്ല' )  then 'UNRESOLVED'  end) as customer_marked_resolution
from  (
            select id,source_channel_id,responder_id,responder_type,question_id,response_value,csat_status,source,created_at,modified_at,version,source_channel_type,rating_input_type,agent_type 
            from crm_octavius.csat
            where date(date_parse(substring(replace(created_at, 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s') + interval '330' minute) >= date('2025-05-16') 
            union 
            select id,source_channel_id,responder_id,responder_type,question_id,response_value,csat_status,source,created_at,modified_at,version,source_channel_type,rating_input_type,agent_type 
            from crm_ntl_octavius.csat
            where date(date_parse(substring(replace(created_at, 'T', ' '), 1, 19), '%Y-%m-%d %H:%i:%s') + interval '330' minute) >= date('2025-05-16') 
            ) csat
inner join (
select conversation_id,sub_ticket_id,ticket_created_time
,lead(cast(ticket_created_time as timestamp), cast(1 as bigint),cast('2100-01-01 00:00:00' as timestamp)) over (partition by conversation_id order by ticket_created_time asc) as next_open_time 
from corp_analytics.cx_sub_ticket_data_model_tbl 
) tea_agg on tea_agg.conversation_id=csat.source_channel_id 
and cast(split_part(replace(csat.modified_at, 'T', ' '), '.', 1) as timestamp) + interval '330' minute between cast(ticket_created_time as timestamp) and next_open_time
where date(cast(split_part(replace(created_at, 'T', ' '), '.', 1) as timestamp) + interval '330' minute) between date(now()+interval '330' minute) - interval '35' day  and date(now()+interval '330' minute) 
group by 1
)

select distinct a.*,b.csat as b_csat,
case when (group_name in ('Grievance Team','Claims Support Emails','manager.experience','GI+HI L2','L2 Team','Escalation','Home Loans - L2','PL-Escalations-RCA','CEO-Escalations','Pl-Social-Media-RCA','UPI L2','BBPS L2')
or a.agent_type='SUPPORT_AGENT_L2') then 'L2' else 'L1' end as L1_L2_Type
from corp_analytics.cx_all_channels_test_tbl a
left join csat_new b on a.ticket_id=b.sub_ticket_id
)
where agent_type in ('SUPPORT_AGENT_L2', 'SUPPORT_AGENT_L1')
group by 1
order by 1 DESC
"""
df = get_sql_df(query)

# Step 3: Connect to the Google Sheet
gsheet_url = "https://docs.google.com/spreadsheets/d/1K74Rj7anxi6z5EKoVQnbCovT2BZMe0eA6NyCdre5Qqs/edit?gid=2023855342#gid=2023855342"
worksheet_name = "Focus Metric Data"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

# Step 4: Clear existing data from the worksheet
gs_reader.clear_sheets(worksheet_name, start_row=1, start_column=1)

# Step 5: Write the new DataFrame to the worksheet (including headers)
gs_reader.set_df_in_sheets(worksheet_name, df, include_headers=True, row=1, col=1)